In [ ]:
import time
import jax
import jax.numpy as jnp
import optax
from flax.training import train_state
from torch.utils.data import DataLoader
from transformers import GPT2Tokenizer
from datasets import load_dataset
from models.gpt2_jax import GPT2JAX


def prepare_dataset(tokenizer, split="train", seq_len=256):
    ds = load_dataset("wikitext", "wikitext-2-raw-v1")[split]

    def encode(ex):
        tok = tokenizer(
            ex["text"],
            truncation=True,
            padding="max_length",
            max_length=seq_len,
        )
        return {
            "input_ids": tok["input_ids"],
            "attention_mask": tok["attention_mask"],
        }

    ds = ds.map(encode, batched=True, remove_columns=["text"],
                load_from_cache_file=True)
    ds.set_format(type="torch", columns=["input_ids", "attention_mask"])
    return ds


def cross_entropy_loss(logits, labels):
    # shift: predict token i+1 from token i
    logits = logits[:, :-1, :]          # (B, T-1, vocab)
    labels = labels[:, 1:]              # (B, T-1)
    loss = optax.softmax_cross_entropy_with_integer_labels(logits, labels)
    return loss.mean()


@jax.jit
def train_step(state, input_ids):
    def loss_fn(params):
        logits = state.apply_fn(
            {"params": params},
            input_ids,
            training=True,
            rngs={"dropout": jax.random.PRNGKey(0)},
        )
        return cross_entropy_loss(logits, input_ids)

    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    state = state.apply_gradients(grads=grads)
    return state, loss


def train_single_tpu(num_epochs=1, bs=16, lr=2e-5):
    print(f"devices: {jax.devices()}")

    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    # init model
    model = GPT2JAX()
    key = jax.random.PRNGKey(0)
    dummy = jnp.ones((1, 256), dtype=jnp.int32)
    params = model.init(key, dummy)["params"]

    total_params = sum(p.size for p in jax.tree_util.tree_leaves(params))
    print(f"GPT-2 parameters: {total_params/1e6:.1f}M")

    tx = optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(lr, eps=1e-5),
    )
    state = train_state.TrainState.create(
        apply_fn=model.apply, params=params, tx=tx
    )

    loader = DataLoader(
        prepare_dataset(tokenizer, "train"),
        batch_size=bs,
        shuffle=True,
        num_workers=0,
    )

    print(f"steps per epoch: {len(loader)}")

    total_samples = 0

    for epoch in range(num_epochs):
        epoch_start = time.perf_counter()
        window_start = time.perf_counter()
        window_samples = 0

        for i, batch in enumerate(loader):
            input_ids = jnp.array(batch["input_ids"].numpy())

            state, loss = train_step(state, input_ids)

            window_samples += bs
            total_samples += bs

            if i % 20 == 0 and i > 0:
                # block_until_ready ensures TPU ops are complete before timing
                jax.block_until_ready(loss)
                elapsed = time.perf_counter() - window_start
                throughput = window_samples / elapsed

                print(
                    f"epoch={epoch} step={i:>5d} loss={loss:.4f} "
                    f"throughput={throughput:.1f} samples/s"
                )

                window_start = time.perf_counter()
                window_samples = 0

        jax.block_until_ready(loss)
        epoch_time = time.perf_counter() - epoch_start
        avg_throughput = total_samples / epoch_time

        print(
            f"\n=== epoch {epoch} done | "
            f"time={epoch_time:.1f}s | "
            f"avg_throughput={avg_throughput:.1f} samples/s ===\n"
        )


if __name__ == "__main__":
    train_single_tpu(num_epochs=1, bs=16, lr=2e-5)

In [ ]:
!pip install flax optax transformers datasets -q

In [ ]:
import jax
import jax.numpy as jnp
import flax
import optax

print(f"JAX version: {jax.__version__}")
print(f"devices: {jax.devices()}")
print(f"device count: {jax.device_count()}")

# confirm it can actually compute
x = jnp.ones((4, 256, 768))
print(f"test tensor device: {x.devices()}")
print(f"test tensor dtype: {x.dtype}")
print(f"flax: {flax.__version__}")
print(f"optax: {optax.__version__}")